# Dataset exploration — Synthetic SMOA & truth 50 bang

Notebook mô tả dữ liệu trước khi train/backtest. Các câu hỏi:
1. Có bao nhiêu curve, quan sát và nhóm synthetic? Độ dài và scale khác nhau thế nào?
2. Truth có cùng lịch quan sát không? Có giá trị âm, số 0, missing hoặc trùng tuần không?
3. Hình dạng và độ lớn khác nhau thế nào giữa các bang và synthetic families?
4. Các đặc điểm này ảnh hưởng gì tới embedding và cách đọc điểm benchmark?

Chỉ đọc CSV gốc; không impute, sửa hoặc xóa quan sát. Đây là phân tích mô tả
trên toàn bộ dữ liệu, không dùng để chọn hyperparameter theo kết quả test.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "experiments/synthetic/generator.py").exists())
sys.path.insert(0, str(ROOT / "experiments/exploration"))
from explore_datasets import (load_inputs, summarize_synthetic, summarize_truth,
                             truth_anomalies, largest_changes, run_exploration)
SYNTHETIC = ROOT / "outputs/reproduction_full/01_synthetic/curves.csv"
MANIFEST = ROOT / "outputs/truth_states/manifest.csv"
OUT = ROOT / "outputs/dataset_exploration"
K, H = 5, 4  # source reference configuration; BO-selected bank may use another k
print("Input: full synthetic + truth_states. Output:", OUT.relative_to(ROOT))

## 1. Đọc dữ liệu và schema
`step` là bước trong cadence gốc, chưa phải ngày lịch. Truth là final truth dùng để chấm điểm.

In [ ]:
synthetic, truth, sources = load_inputs(SYNTHETIC, MANIFEST)
display(pd.DataFrame({"dataset": ["synthetic", "truth"],
                      "rows": [len(synthetic), len(truth)],
                      "series": [synthetic.curve_id.nunique(), truth.series.nunique()]}))
display(synthetic.head())
display(truth.head())

## 2. Synthetic: nhóm, độ dài và phân phối
Thống kê từng curve; phân biệt scale/cadence. Giá trị `1e-10` phản ánh floor của generator, không phải dữ liệu thiếu.

In [ ]:
synstats = summarize_synthetic(synthetic, K, H)
display(synstats.groupby(["curve_type", "scale", "time_cadence"]).agg(
    curves=("curve_id", "size"), median_steps=("observations", "median"),
    floor_values=("floor_values", "sum"), embedding_windows=("embedding_windows", "sum")))
display(synstats[["observations", "min_value", "median_value", "max_value"]].describe())
print("Cửa sổ sau kiểm tra variance, k=5, h=4:", synstats.embedding_windows.sum())
print("Curve sai step sequence:", (~synstats.step_sequence_valid).sum())
print("Curve có metadata không nhất quán:", (~synstats.metadata_consistent).sum())

## 3. Truth: lịch quan sát và chất lượng dữ liệu
Giữ nguyên giá trị âm/số 0. Runner backtest có thể floor ở 1, nhưng EDA hiển thị raw trước xử lý.

In [ ]:
truthstats = summarize_truth(truth)
display(truthstats.groupby(["first_date", "last_date"]).agg(
    states=("series", "size"), min_weeks=("observations", "min"), max_weeks=("observations", "max")))
display(truthstats[["series", "observations", "missing_values", "invalid_dates",
    "duplicate_date_rows", "nonweekly_intervals", "missing_week_slots", "negative_values", "zeros"]])

## 4. Giá trị bất thường và mức thay đổi lớn
Zero không tự động là lỗi. Thay đổi lớn có thể là sóng dịch hoặc thay đổi báo cáo; chỉ đánh dấu để kiểm tra, không tự gọi là outlier để loại bỏ.

In [ ]:
anomalies = truth_anomalies(truth)
display(anomalies[anomalies.value < 0][["series", "target_end_date", "value", "flags"]])
print("Số 0 trong raw truth:", int((truth.value == 0).sum()))
print("Số giá trị sẽ thay đổi khi floor ở 1:", int((truth.value < 1).sum()))
display(largest_changes(truth)[["series", "target_end_date", "previous_value", "value", "change"]])

## 5. Xuất biểu đồ và báo cáo
Có thể chạy lại cell này: chỉ ghi các file dẫn xuất trong thư mục EDA, không thay đổi input. Plot scale normalized chỉ để mô tả; không phải preprocessing của forecast.

In [ ]:
summary = run_exploration(SYNTHETIC, MANIFEST, OUT, K, H)
print(json.dumps(summary, indent=2))

### Hình dạng từng nhóm synthetic
Mỗi curve chia cho peak riêng; tối đa 3 curve mỗi family/scale. Nhãn ghi cadence gốc; trục step không phải cùng một độ dài thời gian cho monthly và weekly.

In [ ]:
display(Image(filename=str(OUT / "figures/01_synthetic_examples.png")))

### Peak và độ dài synthetic
Peak counts và proportion được tách riêng; histogram dùng log10 peak. Các curve ngắn/dài tạo số cửa sổ training khác nhau.

In [ ]:
display(Image(filename=str(OUT / "figures/02_synthetic_distributions.png")))

### Lịch quan sát của 50 bang
Màu xanh là có dòng CSV tại tuần đó, không khẳng định giá trị hợp lệ; bảng audit phía trên kiểm tra giá trị.

In [ ]:
display(Image(filename=str(OUT / "figures/03_truth_calendar.png")))

### Chuỗi số ca raw của bốn bang
Giữ giá trị âm và đánh dấu riêng. Khác biệt magnitude khiến pooled absolute errors có thể bị chi phối bởi bang lớn.

In [ ]:
display(Image(filename=str(OUT / "figures/04_truth_examples.png")))

### Quy mô và thời điểm giữa các bang
Panel trái chỉ dùng tuần có mặt ở mọi bang, clip âm về 0 cho hiển thị và chia cho peak mỗi bang trên chính khoảng chung đó; panel phải dùng peak raw theo toàn lịch từng bang.

In [ ]:
display(Image(filename=str(OUT / "figures/05_truth_scale_and_timing.png")))

## 6. Cách dùng kết quả

- Thư viện mặc định có 18.387 attempts mỗi family; kiểm tra accepted/failed trong metadata, không suy từ số dòng CSV.
- Nhiều floor values tạo cửa sổ phẳng; embedding loại cửa sổ history có variance bằng 0.
- Độ dài/cadence khác nhau làm số cửa sổ mỗi curve khác nhau; chưa tự reweight hay resample.
- Các bang có thể có ngày kết thúc khác nhau: dùng khoảng ngày chung khi benchmark.
- Raw truth có thể có giá trị âm. Ghi rõ khi dùng `--negative-truth clip`; không thay thế bằng dữ liệu as-of.
- Final truth đã có thể được chỉnh sửa sau origin; EDA này chưa chứng minh khả năng dự báo thời gian thực.
- Biểu đồ mô tả full dataset không dùng để tune mô hình hoặc chọn giai đoạn có điểm đẹp.

Báo cáo có thể mở tại `outputs/dataset_exploration/report.md`; các bảng CSV và metadata/checksum ở cùng thư mục.